In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Chácara)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira - Chacara - INEP 35107700 Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    x = str(x)

    # remove qualquer símbolo que não seja número, vírgula ou ponto
    x = re.sub(r"[^0-9,\.]", "", x)

    # caso venha no formato errado tipo "19.448.421.299.999.900"
    # mantemos somente o último grupo decimal
    if x.count(".") > 1:
        # remove TODOS os pontos; eles NÃO representam milhar
        x = x.replace(".", "")

    # agora troca vírgula por ponto
    x = x.replace(",", ".")

    try:
        return float(x)
    except:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)

# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
df["Total_0_9"] = (
    df["v01031_0_4anos"].fillna(0) +
    df["v01032_5_9anos"].fillna(0)
)

# 4. Corrigir renda para 2025 (inflação)

In [5]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [6]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_9"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [7]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_9": "median",
          "populacao_total": "median"
      })
)

In [8]:
# Renomear colunas para refletir que são médias, não totais
df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_9": "mediana_criancas_0_9",
    "populacao_total": "populacao_mediana"
})

# 7. Score final no nível do CEP (correto)

In [9]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_9"]
)

# 8. Ranking final

In [10]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

           CEP                            Bairro  renda_mediana_2025  \
585  05635-050                Jardim Monte Kemel         24449.50200   
382  04719-905  Chácara Santo Antônio (Zona Sul)         32647.68045   
739  05709-040                       Vila Suzana         36272.94825   
413  04729-060                  Jardim Caravelas         19448.42130   
62   04564-900                    Cidade Monções         34184.38485   
253  04660-000                  Jardim Marajoara         29669.06250   
110  04583-909                     Vila Cordeiro         28934.10135   
397  04726-160                     Vila Cruzeiro         24868.25880   
689  05679-050           Jardim Panorama D'Oeste         35331.79650   
574  05634-001                Jardim Monte Kemel         22594.04070   
797  05726-140                      Vila Andrade         19572.22575   
415  04730-000                   Várzea de Baixo         18074.72205   
71   04566-905                    Cidade Monções         34482.8

In [11]:
# Arredondar para 2 casas decimais (padrão monetário)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)

In [12]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("chacara_top_ceps_2025.csv", index=False)

In [13]:
# Lista dos bairros desejados
bairros_desejados = ["Granja Julieta", "Chácara Santo Antônio", "Santo Amaro", "Chácara Flora", "Jardim Santo Amaro", "Vila Andrade"]

# Filtrar diretamente no top_ceps
top_ceps_filtrado = top_ceps[top_ceps["Bairro"].isin(bairros_desejados)]

top_ceps_filtrado

,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_9,populacao_mediana,score_trafego_2025
797,05726-140,Vila Andrade,19572.23,109.0,612.0,2133372.61
346,04709-901,Santo Amaro,25861.42,79.0,505.0,2043052.20
261,04662-902,Santo Amaro,28198.32,67.5,633.5,1903386.61
461,04747-140,Santo Amaro,38211.82,48.0,367.0,1834167.54
777,05717-250,Vila Andrade,24980.56,70.0,436.0,1748639.16
...,...,...,...,...,...,...
354,04710-180,Santo Amaro,6705.14,0.0,42.0,0.00
763,05715-010,Vila Andrade,14777.84,0.0,125.0,0.00
123,04603-004,Santo Amaro,9463.30,0.0,119.0,0.00
125,04604-005,Santo Amaro,14214.95,0.0,150.0,0.00


In [14]:
top_ceps_filtrado.to_csv("chacara_top_ceps_filtrados_2025.csv", index=False)